# AccessMap: Modeling & Retrieval (PMR + Housing)

## Project Description
This is notebook 3 of 3. Notebooks 1 and 2 cleaned PMR (Amsterdam
sidewalks) and Housing (US accessibility labels). Both are saved in
`data/processed/`. This notebook builds the models.

## Why we don't merge PMR and Housing
We checked the coordinates. PMR is Amsterdam only (EPSG:28992). Housing
is spread across the whole US. No overlap. A spatial join would match
Amsterdam sidewalks to houses thousands of miles away. That's noise,
not signal. So we run the same 3 algorithms on each dataset separately,
then compare results at the end.

## Dataset
- **PMR:** `data/processed/pmr_clean.gpkg` (from notebook 01)
- **Housing:** `data/processed/housing_clean.csv` (from notebook 02)

## Project Roadmap

| Step | What we do |
|------|------------|
| 1 | Load both clean datasets |
| 2 | PMR: build an accessibility label, pick features |
| 3 | PMR: Random Forest |
| 4 | Housing: pick features, Random Forest |
| 5 | Compare both Random Forest models |
| 6 | ModernBERT embeddings (Housing address text) |
| 7 | FAISS similarity search (within each dataset) |
| 8 | Export models, embeddings, index |

## Note on the PMR label
PMR's real `wheelchair_accessible` column only has 50 rows filled in,
out of 72,274. All 50 are transit stops, not sidewalks. Too few rows
to train on. So for the 72k sidewalk rows we build our own label,
`pmr_accessible`, from width and curb height. This is our own rule, not
a verified label like Housing's `sidewalk_ok`. Keep this in mind for
the bias write-up.


In [2]:
# Step 1: Load Clean Data (PMR + Housing)
# Just load both files from notebook 01/02. No cleaning here.
import pandas as pd
import geopandas as gpd

PMR_PATH = '../data/processed/pmr_clean.gpkg'
HOUSING_PATH = '../data/processed/housing_clean.csv'

pmr = gpd.read_file(PMR_PATH)
housing = pd.read_csv(HOUSING_PATH)

print(f"PMR shape: {pmr.shape[0]:,} rows x {pmr.shape[1]} columns")
print(f"Housing shape: {housing.shape[0]:,} rows x {housing.shape[1]} columns")

print(f"\nPMR CRS: {pmr.crs}")
minx, miny, maxx, maxy = pmr.total_bounds
print(f"PMR bounding box (meters, EPSG:28992): "
      f"x=[{minx:,.0f}, {maxx:,.0f}], y=[{miny:,.0f}, {maxy:,.0f}]")
print(f"  -> roughly a {maxx-minx:,.0f}m x {maxy-miny:,.0f}m area (Amsterdam)")

print("\nHousing lat/long range:")
print(housing[['lat', 'long']].describe().loc[['min', 'max']].round(2))

# CHANGED: plain-English summary of scale, plus the actual point of
# this step -- confirming no overlap
lat_span = housing['long'].max() - housing['long'].min()
print(f"  -> spans {lat_span:,.0f} degrees longitude, roughly the width "
      f"of the entire US")
print(f"\nNo overlap: PMR is one small area in Amsterdam (meters). "
      f"Housing spans a continent (degrees). Keep the two pipelines separate.")

PMR shape: 72,274 rows x 19 columns
Housing shape: 10,104 rows x 19 columns

PMR CRS: EPSG:28992
PMR bounding box (meters, EPSG:28992): x=[114,084, 122,716], y=[484,488, 487,978]
  -> roughly a 8,632m x 3,491m area (Amsterdam)

Housing lat/long range:
       lat    long
min  25.64 -123.35
max  77.55  166.16
  -> spans 290 degrees longitude, roughly the width of the entire US

No overlap: PMR is one small area in Amsterdam (meters). Housing spans a continent (degrees). Keep the two pipelines separate.


## Step 1 Summary - Load Clean Data

### What we did:
- Loaded `pmr_clean.gpkg` and `housing_clean.csv`. Both are already
  clean from notebooks 01 and 02.

### What we found:
- PMR: 72,274 rows, Amsterdam only (~8,632m x 3,491m area).
- Housing: 10,104 rows, spread across the US (spans 290 degrees
  longitude — roughly the width of the entire continent).
- Both now have 19 columns (18 original + a confidence flag from
  notebook 01/02: `low_confidence_width` for PMR,
  `low_confidence_residential` for Housing).
- No geographic overlap. PMR is one small area measured in meters.
  Housing spans a continent measured in degrees. Confirms we keep the
  two pipelines separate.

### What this means:
- No spatial join is possible or appropriate here. Any comparison
  between PMR and Housing has to happen at the model-result level
  (Step 5), not the row level.

### Next step:
-> Step 2: PMR Accessibility Label + Features


In [3]:
# Step 2: PMR - Accessibility Label + Features
# Build the pmr_accessible label, then pick and encode features.
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import LabelEncoder

pmr = gpd.read_file('../data/processed/pmr_clean.gpkg')

# ---------------------------------------------------------
# 2.1 Our accessibility rule
# ---------------------------------------------------------
width_ok = pmr['obstacle_free_width_float'] >= 0.9
curb_ok = pmr['curb_height_max'].isna() | (pmr['curb_height_max'] <= 0.06)
pmr['pmr_accessible'] = (width_ok & curb_ok).astype(int)

label_counts = pmr['pmr_accessible'].value_counts(normalize=True)
print("Label balance:")
print(f"  Accessible (1):     {label_counts[1]:.1%}  ({(pmr['pmr_accessible']==1).sum():,} rows)")
print(f"  Not accessible (0): {label_counts[0]:.1%}  ({(pmr['pmr_accessible']==0).sum():,} rows)")

# ---------------------------------------------------------
# 2.2 Pick features
# ---------------------------------------------------------
pmr_features = ['length', 'obstacle_free_width_float', 'crossing', 'width_fill']
pmr_model_df = pmr[pmr_features + ['pmr_accessible']].copy()

# CHANGED: name the encoder so Step 8 can save it
crossing_encoder = LabelEncoder()
pmr_model_df['crossing'] = crossing_encoder.fit_transform(
    pmr_model_df['crossing'].astype(str)
)
pmr_model_df = pmr_model_df.fillna(pmr_model_df.median(numeric_only=True))

print(f"\nFeature table shape: {pmr_model_df.shape[0]:,} rows x {pmr_model_df.shape[1]} columns")
pmr_model_df.head()

Label balance:
  Accessible (1):     80.8%  (58,394 rows)
  Not accessible (0): 19.2%  (13,880 rows)

Feature table shape: 72,274 rows x 5 columns


,length,obstacle_free_width_float,crossing,width_fill,pmr_accessible
0,1.99,1.6,0,0.0,1
1,1.99,1.6,0,0.0,1
2,1.99,1.6,0,0.0,1
3,1.99,1.6,0,0.0,1
4,1.99,1.6,0,0.0,1


## Step 2 Summary - PMR Accessibility Label + Features

### What we did:
- Built `pmr_accessible` from width + curb height. The real
  `wheelchair_accessible` column is too small to use (see intro note).
- Picked 4 features with low missing data: `length`,
  `obstacle_free_width_float`, `crossing`, `width_fill`.
- Turned `crossing` (Yes/No) into 0/1. Filled the rest with the
  column median.

### What we found:
- Label balance: 80.8% accessible (58,394 rows), 19.2% not accessible (13,880 rows).
- Feature table: 72,274 rows x 5 columns (4 features + label).

### What this means:
- Good enough balance for Random Forest with `class_weight='balanced'`.
- The label comes from width and curb height. So expect
  `obstacle_free_width_float` to dominate feature importance in Step
  3. That's just our rule showing up again, not a new finding. Say
  this clearly in the Step 3 write-up so it's not overstated.

### Next step:
-> Step 3: PMR Random Forest


In [4]:
# Step 3: PMR - Random Forest
# Train and check Random Forest on the pmr_accessible label.
import pandas as pd
import geopandas as gpd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

pmr = gpd.read_file('../data/processed/pmr_clean.gpkg')
width_ok = pmr['obstacle_free_width_float'] >= 0.9
curb_ok = pmr['curb_height_max'].isna() | (pmr['curb_height_max'] <= 0.06)
pmr['pmr_accessible'] = (width_ok & curb_ok).astype(int)

pmr_features = ['length', 'obstacle_free_width_float', 'crossing', 'width_fill']
pmr_model_df = pmr[pmr_features + ['pmr_accessible']].copy()

# CHANGED: reuse the same crossing_encoder from Step 2, don't create a new one
pmr_model_df['crossing'] = crossing_encoder.transform(
    pmr_model_df['crossing'].astype(str)
)
pmr_model_df = pmr_model_df.fillna(pmr_model_df.median(numeric_only=True))

X = pmr_model_df[pmr_features]
y = pmr_model_df['pmr_accessible']
X_train_pmr, X_test_pmr, y_train_pmr, y_test_pmr = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_pmr = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight='balanced', random_state=42
)
rf_pmr.fit(X_train_pmr, y_train_pmr)
pmr_pred = rf_pmr.predict(X_test_pmr)

acc = accuracy_score(y_test_pmr, pmr_pred)
baseline = y_test_pmr.value_counts(normalize=True).max()

print("PMR Random Forest")
print(f"Accuracy:              {acc:.1%}")
print(f"Majority-class baseline: {baseline:.1%}  (always guessing 'accessible')")
print(f"Improvement over baseline: {(acc - baseline)*100:+.1f} percentage points")
print()
print(classification_report(y_test_pmr, pmr_pred, target_names=['not_accessible', 'accessible']))

pmr_importance = pd.Series(rf_pmr.feature_importances_, index=pmr_features).sort_values(ascending=False)
print("\nFeature importance (higher = more influence on the prediction):")
for feat, score in pmr_importance.items():
    print(f"  {feat:<28} {score:.1%}")

PMR Random Forest
Accuracy:              84.2%
Majority-class baseline: 80.8%  (always guessing 'accessible')
Improvement over baseline: +3.4 percentage points

                precision    recall  f1-score   support

not_accessible       0.55      1.00      0.71      2776
    accessible       1.00      0.81      0.89     11679

      accuracy                           0.84     14455
     macro avg       0.77      0.90      0.80     14455
  weighted avg       0.91      0.84      0.86     14455


Feature importance (higher = more influence on the prediction):
  obstacle_free_width_float    50.6%
  crossing                     25.3%
  length                       12.4%
  width_fill                   11.7%


## Step 3 Summary - PMR Random Forest

### What we did:
- Split 80/20, trained a Random Forest (`n_estimators=200,
  max_depth=10, class_weight='balanced'`) on the 4 PMR features from
  Step 2.

### What we found:
- Accuracy 84.2% vs. majority-class baseline 80.8%. Only +3.4
  percentage points above baseline.
- `not_accessible` recall is 1.00 but precision is only 0.55. Model
  over-flags segments as not accessible, but doesn't miss real ones.
- `obstacle_free_width_float` dominates feature importance, as
  expected, since it's one of the two inputs to the label.

### What this means:
- This mostly checks that our rule from Step 2 is consistent, not
  that the model found something new. Width IS part of the label, so
  of course width predicts it well. Say "our rule-based label behaves
  sensibly", not "the model discovered width matters".
- `crossing` and `length` having any importance at all is the more
  interesting part. Neither one feeds into the label directly.

### Next step:
-> Step 4: Housing Features + Random Forest


In [5]:
# Step 4: Housing - Features + Random Forest
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

housing = pd.read_csv('../data/processed/housing_clean.csv')

housing = housing[housing['sidewalk_ok'].isin(['yes', 'no'])].copy()
housing['target'] = (housing['sidewalk_ok'] == 'yes').astype(int)

housing['state'] = housing['aadress'].str.extract(r',\s*([A-Z]{2})\s*\d{5}')
housing['state'] = housing['state'].fillna('unknown')

house_features = ['house_types', 'house_types:confidence',
                   'residential_yes:confidence', 'state']
h_model_df = housing[house_features + ['target']].copy()
h_model_df['house_types'] = h_model_df['house_types'].fillna('unknown')

# CHANGED: name both encoders so Step 8 can save them
house_types_encoder = LabelEncoder()
h_model_df['house_types'] = house_types_encoder.fit_transform(h_model_df['house_types'])

state_encoder = LabelEncoder()
h_model_df['state'] = state_encoder.fit_transform(h_model_df['state'])

h_model_df = h_model_df.fillna(h_model_df.median(numeric_only=True))

X = h_model_df[house_features]
y = h_model_df['target']
X_train_house, X_test_house, y_train_house, y_test_house = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_housing = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight='balanced', random_state=42
)
rf_housing.fit(X_train_house, y_train_house)
housing_pred = rf_housing.predict(X_test_house)

acc = accuracy_score(y_test_house, housing_pred)
baseline = y_test_house.value_counts(normalize=True).max()

print("Housing Random Forest")
print(f"Accuracy:                {acc:.1%}")
print(f"Majority-class baseline: {baseline:.1%}  (always guessing 'ok')")
print(f"Gap vs. baseline:        {(acc - baseline)*100:+.1f} percentage points "
      f"({'below' if acc < baseline else 'above'} baseline)")
print()
print(classification_report(y_test_house, housing_pred, target_names=['not_ok', 'ok']))

housing_importance = pd.Series(rf_housing.feature_importances_, index=house_features).sort_values(ascending=False)
print("\nFeature importance (higher = more influence on the prediction):")
for feat, score in housing_importance.items():
    print(f"  {feat:<28} {score:.1%}")

Housing Random Forest
Accuracy:                67.4%
Majority-class baseline: 85.3%  (always guessing 'ok')
Gap vs. baseline:        -17.9 percentage points (below baseline)

              precision    recall  f1-score   support

      not_ok       0.21      0.42      0.28       189
          ok       0.88      0.72      0.79      1096

    accuracy                           0.67      1285
   macro avg       0.54      0.57      0.53      1285
weighted avg       0.78      0.67      0.71      1285


Feature importance (higher = more influence on the prediction):
  state                        89.5%
  house_types                  4.3%
  residential_yes:confidence   3.4%
  house_types:confidence       2.8%


## Step 4 Summary - Housing Random Forest

### What we did:
- Filtered to rows with a real `sidewalk_ok` label, pulled `state`
  out of the address, and trained a Random Forest with the same
  settings as PMR's (Step 3), on `house_types`,
  `house_types:confidence`, `residential_yes:confidence`, and `state`.

### What we found:
- Accuracy 67.4%, 17.9 percentage points below the 85.3%
  majority-class baseline. Model is worse than always guessing "ok"
  on raw accuracy.
- Minority class (`not_ok`) recall goes up to 0.42 though, vs. near 0
  for a majority-guess model. Trade-off from `class_weight='balanced'`:
  better at catching bad sidewalks, worse overall accuracy.
- `state` dominates feature importance (89.5%). `house_types` and the
  confidence columns barely matter.

### What this means:
- Don't lead with the accuracy number alone, it looks bad next to the
  baseline. Lead with the trade-off: we chose better recall on bad
  sidewalks over raw accuracy.
- `state` dominating is a bias finding, not just a model result.
  Sidewalk quality varies a lot by state (matches notebook 02's bias
  check -- CA is 24.6% of all rows alone, MO worst at 39% not-ok, UT
  best at 4%). The model is mostly learning "which state is this",
  not learning from house-level features. Say this directly in the
  bias section.

### Next step:
-> Step 5: Compare Both Random Forest Models


In [6]:
# Step 5: Compare Both Random Forest Models
# Side-by-side comparison. This is where "combining" the two
# datasets happens -- comparing results, not merging rows.
# Run after Steps 3 and 4 in the same kernel session.
import pandas as pd

pmr_acc = accuracy_score(y_test_pmr, pmr_pred)
pmr_baseline = y_test_pmr.value_counts(normalize=True).max()
house_acc = accuracy_score(y_test_house, housing_pred)
house_baseline = y_test_house.value_counts(normalize=True).max()

comparison = pd.DataFrame({
    'PMR (pmr_accessible)': {
        'Accuracy': f"{pmr_acc:.1%}",
        'Majority baseline': f"{pmr_baseline:.1%}",
        'Gap vs. baseline': f"{(pmr_acc - pmr_baseline)*100:+.1f}pp",
        'Top feature': pmr_importance.idxmax(),
        'Top feature importance': f"{pmr_importance.max():.1%}",
    },
    'Housing (sidewalk_ok)': {
        'Accuracy': f"{house_acc:.1%}",
        'Majority baseline': f"{house_baseline:.1%}",
        'Gap vs. baseline': f"{(house_acc - house_baseline)*100:+.1f}pp",
        'Top feature': housing_importance.idxmax(),
        'Top feature importance': f"{housing_importance.max():.1%}",
    },
})

print(comparison)

                             PMR (pmr_accessible) Housing (sidewalk_ok)
Accuracy                                    84.2%                 67.4%
Majority baseline                           80.8%                 85.3%
Gap vs. baseline                           +3.4pp               -17.9pp
Top feature             obstacle_free_width_float                 state
Top feature importance                      50.6%                 89.5%


## Step 5 Summary - Comparing the Two Models

### What we did:
- Put PMR's and Housing's accuracy, baseline, and top feature
  side-by-side in one table.

### What we found:
- PMR beats its baseline: 84.2% vs. 80.8% (+3.4pp). Housing does not:
  67.4% vs. 85.3% (-17.9pp), though it trades accuracy for better
  recall on bad sidewalks.
- PMR's top feature is physical (`obstacle_free_width_float`, 50.6%
  importance). Housing's top feature is geographic (`state`, 89.5%
  importance). That difference is the real finding here.

### What this means:
- The two datasets measure accessibility in different ways. PMR
  measures the physical environment directly. Housing measures it
  through crowd labels tied to location. Neither model transfers to
  the other dataset. This backs up the earlier call not to merge them
  -- each model picks up on something dataset-specific, not something
  a merged model should average over.

### Next step:
-> Step 6: ModernBERT Embeddings (Housing address text)


In [7]:
# Step 6: ModernBERT Embeddings (Housing address text)
# Embed Housing's aadress field. PMR has no free text field worth
# embedding (just categories and numbers), so this step is
# Housing-only.
#
# NOTE: this cell downloads a model from Hugging Face the first time
# you run it. Needs internet. Run this locally or in Colab, not in a
# sandbox without internet.
# pip install sentence-transformers  (run once, then restart kernel)
import pandas as pd
import numpy as np
import time
from sentence_transformers import SentenceTransformer

housing = pd.read_csv('../data/processed/housing_clean.csv')
housing = housing[housing['sidewalk_ok'].isin(['yes', 'no'])].copy()

# nomic-ai/modernbert-embed-base needs a "search_document:" prefix on
# the text you embed (see the model card).
addresses = ('search_document: ' + housing['aadress'].astype(str)).tolist()

model = SentenceTransformer('nomic-ai/modernbert-embed-base')

# CHANGED: time the encoding step so we know how long this takes
start = time.time()
address_embeddings = model.encode(addresses, show_progress_bar=True)
elapsed = time.time() - start

# CHANGED: readable, labeled output instead of a raw shape tuple
n_addresses, n_dims = address_embeddings.shape
print(f"Embedded {n_addresses:,} addresses into {n_dims}-dimensional vectors")
print(f"Time taken: {elapsed:.1f} seconds ({n_addresses/elapsed:.0f} addresses/sec)")

np.save('../data/processed/housing_address_embeddings.npy', address_embeddings)
print(f"Saved to: ../data/processed/housing_address_embeddings.npy")

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Batches:   0%|          | 0/201 [00:00<?, ?it/s]

Embedded 6,425 addresses into 768-dimensional vectors
Time taken: 186.1 seconds (35 addresses/sec)
Saved to: ../data/processed/housing_address_embeddings.npy


## Step 6 Summary - ModernBERT Embeddings

### What we did:
- Embedded Housing's `aadress` text with
  `nomic-ai/modernbert-embed-base`. One vector per row.
- Skipped PMR. Its fields are categories and numbers, no free text
  worth embedding.

### What we found:
- Embedded 6,425 addresses into 768-dimensional vectors.
- Took 186.1 seconds (about 35 addresses/second) on this run.

### What this means:
- These vectors capture rough address similarity (same city, similar
  street names, etc). They feed into Step 7's FAISS index.

### Next step:
-> Step 7: FAISS Similarity Search


In [8]:
# Step 7: FAISS Similarity Search
# Build one FAISS index per dataset. Search stays within each
# dataset -- PMR to PMR, Housing to Housing -- since they don't
# share geography (see intro note).
import numpy as np
import pandas as pd
import faiss
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# 7.1 Housing: search over the ModernBERT embeddings from Step 6.
# ---------------------------------------------------------
address_embeddings = np.load('../data/processed/housing_address_embeddings.npy').astype('float32')
housing_index = faiss.IndexFlatL2(address_embeddings.shape[1])
housing_index.add(address_embeddings)

# Example: find the 5 addresses most like row 0.
distances, neighbor_ids = housing_index.search(address_embeddings[:1], 5)

query_address = housing['aadress'].iloc[0]
print(f"Housing: addresses most similar to row 0 ({query_address}):")
for rank, (nid, dist) in enumerate(zip(neighbor_ids[0], distances[0]), start=1):
    print(f"  #{rank}  row {nid:<6} distance={dist:.3f}  {housing['aadress'].iloc[nid]}")

faiss.write_index(housing_index, '../data/processed/housing_faiss.index')

# ---------------------------------------------------------
# 7.2 PMR: no text to embed, so build the index over numeric features
# instead. Still useful -- e.g. "find other segments with a similar
# width/curb/crossing profile to this bad one".
#
# NOTE: the 4 features from Step 3 (length, width, crossing,
# width_fill) are low-cardinality -- many real segments share
# identical values (standardized sidewalk construction). Adding
# curb_height_max (continuous, previously unused here) and
# standardizing all features gives the index a bit more to work with,
# but exact ties will still happen. That's a property of the data,
# not a bug -- see Step 7 Summary.
# ---------------------------------------------------------
pmr_model_df['curb_height_max'] = pmr['curb_height_max']
pmr_model_df['curb_height_missing'] = pmr['curb_height_max'].isna().astype(int)
pmr_model_df['curb_height_max'] = pmr_model_df['curb_height_max'].fillna(
    pmr_model_df['curb_height_max'].median()
)

pmr_retrieval_features = pmr_features + ['curb_height_max', 'curb_height_missing']

scaler = StandardScaler()
pmr_vectors = scaler.fit_transform(
    pmr_model_df[pmr_retrieval_features].to_numpy()
).astype('float32')

pmr_index = faiss.IndexFlatL2(pmr_vectors.shape[1])
pmr_index.add(pmr_vectors)

# Query on a real "not accessible" segment, not row 0 (which happens
# to be one of many exact-duplicate rows -- see Step 7 Summary).
query_idx = pmr_model_df[pmr_model_df['pmr_accessible'] == 0].index[0]
query_vector = pmr_vectors[query_idx:query_idx + 1]
distances, neighbor_ids = pmr_index.search(query_vector, 5)

print(f"\nPMR: segments most similar to row {query_idx} (a 'not accessible' segment):")
print(f"  Query row {query_idx} features:")
for feat in pmr_retrieval_features:
    val = pmr_model_df.loc[query_idx, feat]
    print(f"    {feat:<28} {val}")
for rank, (nid, dist) in enumerate(zip(neighbor_ids[0], distances[0]), start=1):
    tag = " (identical values)" if dist == 0 else ""
    same_label = pmr_model_df.loc[nid, 'pmr_accessible']
    print(f"  #{rank}  row {nid:<6} distance={dist:.3f}  "
          f"pmr_accessible={same_label}{tag}")

faiss.write_index(pmr_index, '../data/processed/pmr_faiss.index')

Housing: addresses most similar to row 0 (4210 Southwest 167th Avenue, Beaverton, OR 97007, USA ):
  #1  row 0      distance=0.000  4210 Southwest 167th Avenue, Beaverton, OR 97007, USA 
  #2  row 3775   distance=0.154  8350 Southwest 168th Avenue, Beaverton, OR 97007, USA
  #3  row 1027   distance=0.268  6360 Southwest 154th Place, Beaverton, OR 97007, USA
  #4  row 4550   distance=0.461  15870 Southwest Bridle Hills Drive, Beaverton, OR 97007, USA
  #5  row 6309   distance=0.464  9350 Southwest Galena Way, Beaverton, OR 97007, USA

PMR: segments most similar to row 145 (a 'not accessible' segment):
  Query row 145 features:
    length                       1.99
    obstacle_free_width_float    0.8
    crossing                     0
    width_fill                   0.0
    curb_height_max              0.06
    curb_height_missing          1
  #1  row 145    distance=0.000  pmr_accessible=0 (identical values)
  #2  row 199    distance=0.000  pmr_accessible=0 (identical values)
  #3  ro

## Step 7 Summary - FAISS Similarity Search

### What we did:
- Built one FAISS index for Housing (ModernBERT address embeddings)
  and one for PMR (numeric features, since PMR has no text).
- For PMR, added `curb_height_max` (continuous) and a
  `curb_height_missing` flag on top of the 4 features from Step 3,
  and standardized all features before indexing, to try to give the
  index more to work with.
- Both indexes stay within their own dataset. No cross-dataset
  lookups. Same reasoning as the decision not to merge PMR and
  Housing.

### What we found:
- Housing: addresses similar to row 0 (Beaverton, OR) returned other
  Beaverton addresses, with distances increasing smoothly from 0.154
  to 0.464. Meaningful similarity ranking.
- PMR: nearest neighbors to a real "not accessible" query row (row
  145) are all exact ties (distance = 0.000), even after adding
  curb_height_max. This isn't fixable by adding more attribute
  features: 74% of PMR rows have no measured curb height (see
  notebook 01) and get the same imputed value, and the remaining
  attributes (length, width, crossing, width_fill) are themselves
  low-cardinality. Many real segments are attribute-identical --
  only their location differs, and location isn't in this index.

### What this means:
- Housing index: "find addresses like this one." Useful for flagging
  a cluster of nearby addresses once one is marked not accessible.
- PMR index: works as a "group finder," not a fine-grained ranker.
  It correctly groups segments sharing the same accessibility
  profile (all 5 matches above are genuinely `pmr_accessible=0`), but
  can't rank similarity within that group. This is a real, documented
  limitation of PMR's attribute schema for retrieval -- not a bug in
  the code, and not fixable by picking a different demo row or adding
  more of the same kind of feature. A true fix would require spatial
  coordinates in the index, which changes what the retrieval is
  doing (location-based, not purely attribute-based).

### Next step:
-> Step 8: Export Models, Embeddings, Index

In [9]:
# Step 8: Export Models, Embeddings, Index
import os
import joblib

os.makedirs('../data/models', exist_ok=True)

# Random Forest models
joblib.dump(rf_pmr, '../data/models/rf_pmr.pkl')
joblib.dump(rf_housing, '../data/models/rf_housing.pkl')

# Also save the encoders/scaler used to build model inputs.
# Without these, new data can't be transformed the same way at
# inference time (e.g. a new address's 'state' string has to map to
# the same integer codes the model was trained on).
joblib.dump(crossing_encoder, '../data/models/crossing_encoder.pkl')
joblib.dump(house_types_encoder, '../data/models/house_types_encoder.pkl')
joblib.dump(state_encoder, '../data/models/state_encoder.pkl')
joblib.dump(scaler, '../data/models/pmr_retrieval_scaler.pkl')

# Grouped by category for a more readable summary
saved_files = {
    "Models": [
        ('../data/models/rf_pmr.pkl', 'PMR Random Forest model'),
        ('../data/models/rf_housing.pkl', 'Housing Random Forest model'),
    ],
    "Encoders & Scalers": [
        ('../data/models/crossing_encoder.pkl', 'PMR crossing LabelEncoder'),
        ('../data/models/house_types_encoder.pkl', 'Housing house_types LabelEncoder'),
        ('../data/models/state_encoder.pkl', 'Housing state LabelEncoder'),
        ('../data/models/pmr_retrieval_scaler.pkl', 'PMR retrieval StandardScaler'),
    ],
    "Embeddings & Indexes": [
        ('../data/processed/housing_address_embeddings.npy', 'Housing ModernBERT embeddings (Step 6)'),
        ('../data/processed/housing_faiss.index', 'Housing FAISS index (Step 7)'),
        ('../data/processed/pmr_faiss.index', 'PMR FAISS index (Step 7)'),
    ],
}

n_total = sum(len(v) for v in saved_files.values())
n_missing = sum(
    1 for group in saved_files.values() for path, _ in group if not os.path.exists(path)
)

print("*" * 100)
print("STEP 8 EXPORT SUMMARY")
print("*" * 100)
for group_name, files in saved_files.items():
    print(f"\n{group_name}:")
    for path, desc in files:
        status = "OK" if os.path.exists(path) else "MISSING"
        print(f"  [{status}] {path:<45} {desc}")
print("-" * 100)
print(f"Total files: {n_total}   |   Missing: {n_missing}")
print("*" * 100)

****************************************************************************************************
STEP 8 EXPORT SUMMARY
****************************************************************************************************

Models:
  [OK] ../data/models/rf_pmr.pkl                     PMR Random Forest model
  [OK] ../data/models/rf_housing.pkl                 Housing Random Forest model

Encoders & Scalers:
  [OK] ../data/models/crossing_encoder.pkl           PMR crossing LabelEncoder
  [OK] ../data/models/house_types_encoder.pkl        Housing house_types LabelEncoder
  [OK] ../data/models/state_encoder.pkl              Housing state LabelEncoder
  [OK] ../data/models/pmr_retrieval_scaler.pkl       PMR retrieval StandardScaler

Embeddings & Indexes:
  [OK] ../data/processed/housing_address_embeddings.npy Housing ModernBERT embeddings (Step 6)
  [OK] ../data/processed/housing_faiss.index         Housing FAISS index (Step 7)
  [OK] ../data/processed/pmr_faiss.index             PMR FAIS

## Step 8 Summary - Export Models, Embeddings, Index

**What we did:**
- Saved both Random Forest models (`rf_pmr.pkl`, `rf_housing.pkl`) to `data/models/`.
- Saved the encoders/scaler used to build model inputs (`crossing_encoder.pkl`, `house_types_encoder.pkl`, `state_encoder.pkl`, `pmr_retrieval_scaler.pkl`), so new data can be transformed the same way at inference time.
- Embeddings and FAISS indexes were already saved in Steps 6-7; this step just verifies and reports on them alongside everything else.
- Grouped the export summary into three categories (Models, Encoders & Scalers, Embeddings & Indexes) with an existence check per file, so a broken pipeline run would surface a `MISSING` flag immediately instead of failing silently downstream.

**What we found:**
- All 9 artifacts saved successfully: 2 models, 4 encoders/scaler, 1 embeddings file, 2 FAISS indexes.
- 0 missing files across all three categories.

**What this means:**
- The pipeline runs end to end. Everything downstream (a demo app, further analysis, or scoring new addresses/segments) can now load these files instead of retraining or re-fitting encoders from scratch.

**Next step:**
→ Fine-tune once the full pipeline runs end to end. Per the plan: finish all 3 notebooks first, then fine-tune.